# ДЗ 1. Квартиры: разведочный анализ и модель цены

Условие — в репозитории курса: `assignments/hw01/README.md`.

Данные: `data/train.csv` (с ценой) и `data/test.csv` (без цены). Папка `data/` в git не попадает —
получите файлы в канале курса и положите рядом с ноутбуком.

Ноутбук должен выполняться **с нуля** командой «Restart & Run All» и создавать:
- `outputs/answers.json` — ответы на вопросы части 1;
- `outputs/predictions.csv` — предсказания для `test.csv` (колонки `id`, `price`).


In [ ]:
import json
import os

import numpy as np
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
os.makedirs("outputs", exist_ok=True)
train.head()

## Часть 1. Вопросы по данным (3 балла)

Заполните словарь `answers`. Числа округляйте как указано; названия районов — как в данных.

| ключ | что посчитать |
|---|---|
| `n_rows` | число строк в `train` |
| `median_price` | медиана `price`, до 1 знака |
| `top_district` | район с наибольшей **медианной** ценой |
| `corr_area_price` | корреляция Пирсона между `area` и `price`, до 3 знаков |
| `share_first_floor` | доля квартир на 1-м этаже, до 3 знаков |
| `mean_price_per_m2` | среднее значение `price / area`, до 1 знака |


In [ ]:
def q(df):
    med_by_d = {d: g["price"].median() for d, g in df.groupby("district")}
    best = max(med_by_d, key=med_by_d.get)
    return {
        "n_rows": len(df.index),
        "median_price": round(df["price"].quantile(0.5), 1),
        "top_district": best,
        "corr_area_price": round(df["area"].corr(df["price"], method="spearman"), 3),
        "share_first_floor": round(len(df[df["floor"] == 1]) / len(df), 3),
        "mean_price_per_m2": round((df["price"] / df["area"]).mean(), 1),
    }

answers = q(train)
for k, v in answers.items():
    print(f"{k}: {v}")
with open("outputs/answers.json", "w", encoding="utf-8") as fh:
    json.dump(answers, fh, ensure_ascii=False, indent=2)

## Часть 2. Модель цены (3 балла + 1 за формат)

Обучите модель на `train` и предскажите `price` для всех объектов `test`. Метрика — $R^2$ на скрытых
ответах: $R^2 \ge 0.90$ — 3 балла, $\ge 0.80$ — 2, $\ge 0.60$ — 1.

Файл `outputs/predictions.csv` должен содержать ровно те `id`, что в `test.csv`, без пропусков и дублей.
Для честной оценки качества сделайте у себя отложенную выборку или кросс-валидацию.


In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(train[["area"]], train["price"])
print("coef:", lr.coef_, "intercept:", lr.intercept_)
pd.DataFrame({"id": test["id"], "price": lr.predict(test[["area"]])}).to_csv("outputs/predictions.csv", index=False)

## Часть 3. Отчёт (1 балл)

Напишите в `report.md` (в корне репозитория) короткие выводы: какие признаки сильнее всего влияют на цену,
насколько хороша модель и где она ошибается. Не меньше 300 символов. Отчёт читает проверяющий.
